# Poker HUD pessoal — PokerStars

Objetivo: ler os hand histories do PokerStars (arquivos `.txt`) e calcular, por
jogador:

- **VPIP** — % de mãos em que o jogador colocou dinheiro voluntariamente no pot pré-flop
- **PFR** — % de mãos em que o jogador deu raise no pré-flop
- **3Bet%** — das vezes em que teve a chance de re-levantar um raise, quantas ele fez
- **Fold to 3Bet%** — das vezes em que abriu o pote e foi re-levantado, quantas desistiu

O notebook está dividido em blocos pequenos: cada um faz uma coisa, com uma
célula de teste logo depois pra você ver o resultado antes de seguir pro
próximo passo. No final apontamos pra sua pasta real de hand history.


In [ ]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd


## Passo 1 — Como é um Hand History do PokerStars

Cada mão é um bloco de texto assim (pode haver dezenas de milhares desses
blocos, um atrás do outro, em cada arquivo):

```
PokerStars Hand #220000000001: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:00:00 ET
Table 'Rio III' 6-max Seat #1 is the button
Seat 1: Alice ($10.00 in chips)
Seat 2: Bob ($10.00 in chips)
Seat 3: Carol ($10.00 in chips)
Bob: posts small blind $0.05
Carol: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [Ah Kd]
Alice: folds
Hero: raises $0.20 to $0.30
Bob: folds
Carol: calls $0.20
*** FLOP *** [2h 7d Jc]
...
*** SUMMARY ***
...
```

Pontos importantes pro parser:

- `Seat N: Nome ($X in chips)` — diz quem estava sentado na mão (nosso
  denominador de "mãos jogadas" pra cada jogador).
- A seção pré-flop fica entre `*** HOLE CARDS ***` e o próximo marcador
  `*** ... ***` (FLOP, SUMMARY, etc).
- Ações seguem o padrão `Nome: ação` — `folds`, `checks`, `calls`, `bets`,
  `raises $X to $Y`.
- Postar blind **não** conta como ação voluntária pro VPIP.

Vamos criar um exemplo pequeno pra testar o parser antes de usar dados reais.


In [ ]:
SAMPLE_HAND_HISTORY = """
PokerStars Hand #220000000001: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:00:00 ET
Table 'Rio III' 6-max Seat #1 is the button
Seat 1: Alice ($10.00 in chips)
Seat 2: Bob ($10.00 in chips)
Seat 3: Carol ($10.00 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.00 in chips)
Seat 6: Frank ($10.00 in chips)
Bob: posts small blind $0.05
Carol: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [Ah Kd]
Dave: folds
Hero: raises $0.20 to $0.30
Frank: folds
Alice: folds
Bob: folds
Carol: calls $0.20
*** FLOP *** [2h 7d Jc]
Carol: checks
Hero: bets $0.40
Carol: folds
*** SUMMARY ***
Total pot $0.75 | Rake $0.03

PokerStars Hand #220000000002: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:05:00 ET
Table 'Rio III' 6-max Seat #2 is the button
Seat 1: Alice ($9.70 in chips)
Seat 2: Bob ($9.95 in chips)
Seat 3: Carol ($9.40 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($10.00 in chips)
Carol: posts small blind $0.05
Dave: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [2c 3c]
Hero: folds
Frank: raises $0.30 to $0.30
Alice: folds
Bob: raises $0.90 to $1.00
Carol: folds
Dave: folds
Frank: folds
Uncalled bet ($0.70) returned to Bob
*** SUMMARY ***
Total pot $0.55 | Rake $0.02

PokerStars Hand #220000000003: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:10:00 ET
Table 'Rio III' 6-max Seat #3 is the button
Seat 1: Alice ($9.70 in chips)
Seat 2: Bob ($10.65 in chips)
Seat 3: Carol ($9.40 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($10.00 in chips)
Dave: posts small blind $0.05
Hero: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [7h 2s]
Frank: raises $0.30 to $0.30
Alice: calls $0.30
Bob: folds
Carol: folds
Dave: folds
Hero: folds
*** FLOP *** [9h 4d Kc]
Frank: bets $0.40
Alice: folds
Uncalled bet ($0.40) returned to Frank
*** SUMMARY ***
Total pot $1.05 | Rake $0.04

PokerStars Hand #220000000004: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:15:00 ET
Table 'Rio III' 6-max Seat #4 is the button
Seat 1: Alice ($9.30 in chips)
Seat 2: Bob ($10.65 in chips)
Seat 3: Carol ($9.40 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($9.60 in chips)
Hero: posts small blind $0.05
Frank: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [Qc Qd]
Alice: raises $0.30 to $0.30
Bob: folds
Carol: calls $0.30
Dave: folds
Hero: folds
Frank: folds
*** FLOP *** [3h 8d 2c]
Alice: bets $0.40
Carol: raises $1.00 to $1.20
Alice: folds
Uncalled bet ($0.80) returned to Carol
*** SUMMARY ***
Total pot $1.15 | Rake $0.04

PokerStars Hand #220000000005: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:20:00 ET
Table 'Rio III' 6-max Seat #5 is the button
Seat 1: Alice ($9.30 in chips)
Seat 2: Bob ($10.65 in chips)
Seat 3: Carol ($8.20 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($9.60 in chips)
Frank: posts small blind $0.05
Alice: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [8h 8s]
Bob: folds
Carol: folds
Dave: folds
Hero: folds
Frank: raises $0.30 to $0.30
Alice: calls $0.20
*** FLOP *** [Ks 9d 4h]
Alice: checks
Frank: bets $0.40
Alice: calls $0.40
*** TURN *** [Ks 9d 4h] [2c]
Alice: checks
Frank: bets $0.90
Alice: folds
Uncalled bet ($0.90) returned to Frank
*** SUMMARY ***
Total pot $1.55 | Rake $0.06

PokerStars Hand #220000000006: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:25:00 ET
Table 'Rio III' 6-max Seat #6 is the button
Seat 1: Alice ($9.30 in chips)
Seat 2: Bob ($10.65 in chips)
Seat 3: Carol ($7.80 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($9.10 in chips)
Alice: posts small blind $0.05
Bob: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [Kh Kc]
Carol: folds
Dave: folds
Hero: folds
Frank: folds
Alice: folds
Bob: checks
*** FLOP *** [Ah 8d 3c]
Bob: bets $0.20
Carol: folds
*** SUMMARY ***
Total pot $0.35 | Rake $0.01

PokerStars Hand #220000000007: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:30:00 ET
Table 'Rio III' 6-max Seat #1 is the button
Seat 1: Alice ($9.30 in chips)
Seat 2: Bob ($10.45 in chips)
Seat 3: Carol ($7.80 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($9.10 in chips)
Bob: posts small blind $0.05
Carol: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [4h 4c]
Dave: folds
Hero: folds
Frank: raises $0.30 to $0.30
Alice: folds
Bob: calls $0.25
Carol: folds
*** FLOP *** [Ks 9d 4h]
Frank: bets $0.40
Bob: calls $0.40
*** TURN *** [Ks 9d 4h] [2c]
Frank: bets $0.90
Bob: calls $0.90
*** RIVER *** [Ks 9d 4h 2c] [7s]
Frank: bets $1.50
Bob: calls $1.50
*** SHOW DOWN ***
Frank: shows [Ac Kd] (a pair of Kings)
Bob: mucks hand
Frank collected $6.30 from pot
*** SUMMARY ***
Total pot $6.30 | Rake $0.25
Board [Ks 9d 4h 2c 7s]
Seat 1: Alice (button) folded before Flop
Seat 2: Bob (small blind) mucked [2h 3h]
Seat 3: Carol (big blind) folded before Flop
Seat 4: Dave folded before Flop
Seat 5: Hero folded before Flop
Seat 6: Frank showed [Ac Kd] and won ($6.30) with a pair of Kings
PokerStars Hand #220000000008: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:35:00 ET
Table 'Rio III' 6-max Seat #1 is the button
Seat 1: Alice ($9.30 in chips)
Seat 2: Bob ($10.65 in chips)
Seat 3: Carol ($7.80 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($9.10 in chips)
Bob: posts small blind $0.05
Carol: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [4h 4c]
Dave: folds
Hero: folds
Frank: raises $0.30 to $0.30
Alice: raises $0.90 to $0.90
Bob: folds
Carol: folds
Frank: raises $2.00 to $2.00
Alice: folds
Uncalled bet ($1.10) returned to Frank
*** SUMMARY ***
Total pot $1.40 | Rake $0.05
Seat 1: Alice folded before Flop
Seat 2: Bob (small blind) folded before Flop
Seat 3: Carol (big blind) folded before Flop
Seat 4: Dave folded before Flop
Seat 5: Hero folded before Flop
Seat 6: Frank collected ($1.40)

PokerStars Hand #220000000009: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:40:00 ET
Table 'Rio III' 6-max Seat #2 is the button
Seat 1: Alice ($8.40 in chips)
Seat 2: Bob ($10.65 in chips)
Seat 3: Carol ($7.80 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.10 in chips)
Seat 6: Frank ($11.00 in chips)
Carol: posts small blind $0.05
Dave: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [Kh Kc]
Hero: raises $0.30 to $0.30
Frank: calls $0.30
Alice: folds
Bob: folds
Carol: folds
Dave: folds
*** FLOP *** [2h 7c Jd]
Hero: bets $0.40
Frank: folds
Uncalled bet ($0.40) returned to Hero
*** SUMMARY ***
Total pot $0.95 | Rake $0.03
Board [2h 7c Jd]
Seat 1: Alice folded before Flop
Seat 2: Bob folded before Flop
Seat 3: Carol (small blind) folded before Flop
Seat 4: Dave (big blind) folded before Flop
Seat 5: Hero collected ($0.95)
Seat 6: Frank folded on the Flop
PokerStars Hand #220000000010: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:45:00 ET
Table 'Rio III' 6-max Seat #6 is the button
Seat 1: Alice ($10.00 in chips)
Seat 2: Bob ($10.00 in chips)
Seat 3: Carol ($10.00 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.00 in chips)
Seat 6: Frank ($10.00 in chips)
Alice: posts small blind $0.05
Bob: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [Ks Qs]
Carol: folds
Dave: folds
Hero: raises $0.20 to $0.30
Frank: folds
Alice: raises $1.00 to $1.10
Bob: folds
Hero: folds
Uncalled bet ($0.80) returned to Alice
*** SUMMARY ***
Total pot $0.75 | Rake $0.02
Seat 1: Alice (small blind) collected ($0.73)
Seat 2: Bob (big blind) folded before Flop
Seat 3: Carol folded before Flop (didn't bet)
Seat 4: Dave folded before Flop (didn't bet)
Seat 5: Hero folded before Flop
Seat 6: Frank (button) folded before Flop (didn't bet)

PokerStars Hand #220000000011: Hold'em No Limit ($0.05/$0.10 USD) - 2024/06/01 20:50:00 ET
Table 'Rio III' 6-max Seat #4 is the button
Seat 1: Alice ($10.00 in chips)
Seat 2: Bob ($10.00 in chips)
Seat 3: Carol ($10.00 in chips)
Seat 4: Dave ($10.00 in chips)
Seat 5: Hero ($10.00 in chips)
Seat 6: Frank ($10.00 in chips)
Hero: posts small blind $0.05
Frank: posts big blind $0.10
*** HOLE CARDS ***
Dealt to Hero [7d 2h]
Alice: folds
Bob: folds
Carol: folds
Dave: raises $0.15 to $0.25
Hero: folds
Frank: calls $0.15
*** FLOP *** [Ah 9s 3d]
Frank: checks
Dave: bets $0.30
Frank: folds
Uncalled bet ($0.30) returned to Dave
*** SUMMARY ***
Total pot $0.55 | Rake $0.02
Board [Ah 9s 3d]
Seat 1: Alice folded before Flop (didn't bet)
Seat 2: Bob folded before Flop (didn't bet)
Seat 3: Carol folded before Flop (didn't bet)
Seat 4: Dave (button) collected ($0.53)
Seat 5: Hero (small blind) folded before Flop
Seat 6: Frank (big blind) folded on the Flop
"""

# Cada arquivo real tem várias mãos concatenadas. Separamos usando o
# cabeçalho "PokerStars Hand #" como marcador de início de cada uma.
HAND_SPLIT_RE = re.compile(r"(?=^PokerStars (?:Hand|Zoom Hand) #)", re.MULTILINE)

sample_hands = [h for h in HAND_SPLIT_RE.split(SAMPLE_HAND_HISTORY) if h.strip().startswith("PokerStars")]
print(f"{len(sample_hands)} mãos encontradas no exemplo")
print(sample_hands[0][:200], "...")


As mãos 3 e 4 do exemplo já têm ação de flop, pra gente testar as stats de
continuation bet mais adiante:

- **Mão 3**: Frank abre pré-flop, Alice paga. No flop, Frank dá cbet, Alice
  desiste → testa **Cbet Flop** e **Fold to Cbet Flop**.
- **Mão 4**: Alice abre pré-flop, Carol paga. No flop, Alice dá cbet, Carol
  dá raise, Alice desiste → testa **Cbet Raise (raise vs cbet)** e
  **Fold to Cbet Raise**.
- **Mão 5**: Frank abre pré-flop, Alice paga. Frank dá cbet no flop, Alice
  paga. No turn, Frank dá cbet de novo (segunda barrel), Alice desiste →
  testa **Cbet Turn** e **Fold to Cbet Turn**.
- **Mão 6**: Bob vê o flop de graça (ninguém abriu) e aposta — mão termina
  no flop, sem showdown → testa que "viu o flop" e "foi a showdown" são
  coisas diferentes.
- **Mão 7**: Frank abre, Bob paga e acompanha até o river; no showdown Frank
  mostra o par maior e ganha → testa **Went to Showdown** e **Win at
  Showdown**, além de alimentar a **Agressão pós-flop** (Frank só apostou,
  Bob só pagou).
- **Mão 8**: Frank abre, Alice dá 3bet, Frank dá 4bet e Alice desiste →
  testa **4Bet%** e **Fold to 4Bet%**.
- **Mão 9**: Hero abre, Frank paga, os dois veem o flop, Hero aposta e Frank
  desiste (sem showdown) → testa **WWSF** (ganhou sem showdown).
- **Mão 10**: Hero abre do CO (steal), Alice no small blind dá 3bet →
  testa **Resteal do SB** (3bet contra steal). O Bob (BB) age depois do 3bet,
  então *não* conta como tendo enfrentado o steal.
- **Mão 11**: Dave abre do botão (steal), Hero no small blind desiste e
  Frank no big blind paga → testa **Fold to Steal do SB** e a defesa do BB.


## Passo 2 — Extraindo os jogadores da mão

Primeiro, algo simples: pegar quem estava sentado na mesa naquela mão (linhas
`Seat N: Nome (...)`). Isso vira o denominador de "mãos jogadas" de cada um.


In [ ]:
SEAT_RE = re.compile(r"^Seat \d+: (.+?) \(\$?[\d,\.]+ in chips\)")

def get_players_in_hand(hand_text):
    players = set()
    for line in hand_text.splitlines():
        if "is sitting out" in line:
            continue
        m = SEAT_RE.match(line.strip())
        if m:
            players.add(m.group(1).strip())
    return players

# teste
get_players_in_hand(sample_hands[0])


## Passo 2b — Descobrindo a posição de cada jogador

Pra calcular stats **por posição** (VPIP/PFR/RFI por posição, steal, defesa dos
blinds) precisamos saber onde cada jogador estava sentado em relação ao botão.
O hand history dá duas pistas: a linha `Seat #N is the button` e o número do
assento de cada jogador (`Seat N: Nome ...`).

A lógica: pegamos só os jogadores **ativos** (que não estão `sitting out`),
ordenamos por número de assento, achamos o botão, e percorremos no sentido
horário: **BTN → SB → BB → UTG → ... → CO**.

- Com 6 jogadores: UTG, MP (o "hijack"), CO.
- Com 5: UTG, CO. Com 4: só CO. Com 7+ jogadores: UTG, vários MP, CO.
- **Heads-up (menos de 3 jogadores ativos) não tem posição** — essas mãos
  simplesmente não entram nas stats por posição.


In [ ]:
BUTTON_RE = re.compile(r"Seat #(\d+) is the button")
SEAT_NUM_RE = re.compile(r"^Seat (\d+): (.+?) \(\$?[\d,\.]+ in chips\)")

POSITIONS = ["UTG", "MP", "CO", "BTN", "SB", "BB"]
STEAL_POSITIONS = ("CO", "BTN", "SB")   # posições de onde um raise "first in" é um steal


def compute_positions(hand_text):
    """Retorna {jogador: posição} para os jogadores ativos da mão.

    Posições: BTN, SB, BB, UTG, MP, CO (MP = tudo entre UTG e CO).
    Mãos com menos de 3 jogadores ativos (heads-up) retornam {}.
    """
    m = BUTTON_RE.search(hand_text)
    if not m:
        return {}
    button_seat = int(m.group(1))

    seats = []
    for line in hand_text.splitlines():
        if "is sitting out" in line:
            continue
        sm = SEAT_NUM_RE.match(line.strip())
        if sm:
            seats.append((int(sm.group(1)), sm.group(2).strip()))
    seats.sort()

    n = len(seats)
    if n < 3:
        return {}

    # índice do botão: último assento ativo com número <= assento do botão
    candidatos = [i for i, (num, _) in enumerate(seats) if num <= button_seat]
    btn_idx = candidatos[-1] if candidatos else n - 1

    # ordem no sentido horário a partir do botão: BTN, SB, BB, UTG, ..., CO
    ordenados = [seats[(btn_idx + k) % n][1] for k in range(n)]

    resto = n - 3  # jogadores depois do BB
    if resto == 0:
        rotulos_resto = []
    elif resto == 1:
        rotulos_resto = ["CO"]
    elif resto == 2:
        rotulos_resto = ["UTG", "CO"]
    else:
        rotulos_resto = ["UTG"] + ["MP"] * (resto - 2) + ["CO"]

    rotulos = ["BTN", "SB", "BB"] + rotulos_resto
    return dict(zip(ordenados, rotulos))


# teste: mão 7 (botão no assento 1) -> Alice BTN, Bob SB, Carol BB, Dave UTG, Hero MP, Frank CO
compute_positions(sample_hands[6])


## Passo 3 — A lógica principal: lendo as ações pré-flop

Aqui está o núcleo do parser. A ideia é percorrer as ações do pré-flop em
ordem, mantendo um contador `raise_count`:

- `raise_count == 0` → ninguém deu raise ainda (só blinds).
- Quando alguém dá o **primeiro** raise (`raise_count` vai de 0 para 1),
  esse é o "open" — conta como **PFR** pra ele.
- Se, com `raise_count == 1`, outro jogador (que não foi quem abriu) age —
  seja qual for a ação — isso é uma **oportunidade de 3bet** pra ele. Se a
  ação dele for `raises`, isso é um **3bet** de fato (`raise_count` vira 2).
- Quando o 3bet acontece, quem tinha aberto (o "opener") passa a ter
  **enfrentado um 3bet**. A próxima ação dele (fold, call ou 4bet) decide se
  conta como **Fold to 3Bet**.
- **VPIP** é simplesmente: o jogador deu `calls`, `bets` ou `raises` no
  pré-flop (postar blind sozinho não conta).

O **4bet** segue exatamente o mesmo padrão, um nível acima: com
`raise_count == 2` (já existe um 3bet), quem age e não foi o 3bettor tem uma
**oportunidade de 4bet**; se der raise, é um **4bet** (`raise_count` vira 3).
Quem fez o 3bet passa a ter **enfrentado um 4bet**, e a reação dele (fold)
conta como **Fold to 4Bet**.

Squeeze plays e potes com 5bet+ são tratados de forma simplificada — o foco
é o caso mais comum (abriu → alguém 3beta → opener reage → 4bet).

A mesma ideia se repete no flop, pra calcular as stats de continuation bet:

- Identificamos o **agressor pré-flop** (quem deu o último raise que não foi
  re-levantado). Se a mão chega ao flop, ele teve a **oportunidade de cbet**.
- Olhamos a *primeira vez que é a vez dele agir* no flop: se ainda ninguém
  apostou e ele aposta, isso é o **cbet**. Se alguém já apostou antes dele
  (um "donk bet"), não contamos como oportunidade de cbet — ele nunca teve a
  chance de apostar primeiro.
- Quem enfrenta esse cbet (qualquer jogador que age depois, exceto quem
  apostou) tem uma **oportunidade de reagir ao cbet**: se ele foldar, conta
  pra **Fold to Cbet Flop**; se ele der raise, conta pra **Cbet Raise
  (raise vs cbet)**.
- Se o cbet for raised, o próprio cbettor enfrenta essa decisão — se ele
  foldar, conta pra **Fold to Cbet Raise**.

O turn e o river funcionam exatamente igual, só muda quem é "o agressor da
rua anterior": no turn, não é mais o agressor do pré-flop, e sim quem fechou a
aposta no flop (o cbettor do flop, ou quem re-levantou o cbet, se foi
raised); no river, quem fechou a aposta no turn. Por isso escrevemos essa
lógica uma vez como uma função reutilizável (`parse_street_cbet`) e a
chamamos três vezes (flop, turn, river), encadeando o "agressor da rua" de
uma pra outra. Cada rua gera as mesmas 4 stats: cbet, fold to cbet, raise vs
cbet e fold to cbet raise.

**RFI, steal e defesa dos blinds** (usam a posição do Passo 2b):

- **RFI (raise first in)**: na *primeira ação* do jogador, se ninguém entrou
  no pote antes (todos foldaram — sem limp nem raise), ele teve uma
  oportunidade de RFI; se deu raise, é um RFI. Calculamos por posição.
- **ATS (attempt to steal)**: RFI vindo do **CO, BTN ou SB** (as posições de
  "roubo dos blinds"), dividido pelas oportunidades nessas mesmas posições.
- **Fold to Steal / Resteal**: SB e BB, na primeira ação, enfrentando um
  steal (um único raise, feito por um steal). Se foldam, é **fold to steal**;
  se dão raise, é um **resteal** (3bet contra steal). Se alguém já deu 3bet
  antes deles agirem, não conta como "enfrentou o steal".
- **VPIP e PFR por posição**: o mesmo VPIP/PFR de antes, só que separado pela
  posição que o jogador ocupava naquela mão.

Por fim, mais quatro stats:

- **Agressão pós-flop (AF)**: `(bets + raises) / calls`, somando as ações em
  flop, turn e river. Não é uma porcentagem — é uma razão (ex: `2.0` significa
  duas ações agressivas pra cada call). Fica `None` se o jogador nunca deu
  call pós-flop nas mãos observadas (divisão por zero).
- **Went to Showdown (WTSD%)**: das mãos em que o jogador **viu o flop**,
  em quantas ele chegou ao showdown (identificamos por uma linha do tipo
  `Seat N: Nome showed [...] ...` ou `Seat N: Nome mucked [...]` na seção
  `*** SUMMARY ***` do hand history).
- **Win at Showdown (W$SD%)**: das mãos em que ele chegou ao showdown, em
  quantas ele ganhou (linha `Seat N: Nome showed [...] and won ($X) with ...`).
- **Won When Saw Flop (WWSF%)**: das mãos em que o jogador **viu o flop**, em
  quantas ele ganhou o pote — com ou sem showdown (linha `... and won ...` ou
  `Seat N: Nome collected ($X)`). Diferente do W$SD, aqui contam também as
  vitórias em que todo mundo desistiu.


In [ ]:
ACTION_RE = re.compile(r"^(.+?): (folds|checks|calls|bets|raises)\b")

def parse_street_cbet(street_lines, prior_aggressor):
    """
    Aplica a lógica de cbet/fold-to-cbet/raise-vs-cbet/fold-to-cbet-raise
    numa única rua pós-flop (flop, turn ou river).

    prior_aggressor: quem foi o agressor da rua anterior (é quem tem a
        "oportunidade de cbet" nesta rua).

    Retorna (resultado, agressor_desta_rua) — o segundo valor serve pra
    encadear a chamada seguinte (ex: agressor do flop -> prior_aggressor do turn).
    """
    result = {
        "opportunity": set(), "cbet": set(),
        "faced_cbet_opp": set(), "fold_to_cbet": set(),
        "raise_vs_cbet": set(), "faced_raise_vs_cbet": set(),
        "fold_to_cbet_raise": set(),
    }
    street_aggressor = None
    if not street_lines:
        return result, street_aggressor

    bet_occurred = False
    cbet_bettor = None
    aggressor_had_first_chance = False
    already_faced_cbet = set()
    waiting_reactor = None

    for line in street_lines:
        m = ACTION_RE.match(line.strip())
        if not m:
            continue
        player, action = m.group(1).strip(), m.group(2)

        if action in ("bets", "raises"):
            street_aggressor = player  # quem fecha a aposta nesta rua (usado na próxima rua)

        # primeira vez que é a vez do agressor da rua anterior agir, sem ninguém ter apostado antes: oportunidade de cbet
        if prior_aggressor and not bet_occurred and player == prior_aggressor and not aggressor_had_first_chance:
            aggressor_had_first_chance = True
            result["opportunity"].add(player)
            if action == "bets":
                bet_occurred = True
                cbet_bettor = player
                result["cbet"].add(player)
            continue

        if not bet_occurred:
            if action == "bets":
                bet_occurred = True
                cbet_bettor = player  # bet de outro jogador (donk bet) - não conta como cbet
            continue

        # já existe uma aposta na mesa
        if cbet_bettor == prior_aggressor:
            if player != cbet_bettor and player not in already_faced_cbet:
                already_faced_cbet.add(player)
                result["faced_cbet_opp"].add(player)
                if action == "folds":
                    result["fold_to_cbet"].add(player)
                elif action == "raises":
                    result["raise_vs_cbet"].add(player)
                    result["faced_raise_vs_cbet"].add(cbet_bettor)
                    waiting_reactor = cbet_bettor

            if waiting_reactor == cbet_bettor and player == cbet_bettor:
                if action == "folds":
                    result["fold_to_cbet_raise"].add(player)
                waiting_reactor = None

    return result, street_aggressor


def extract_street_lines(lines, marker):
    """Isola as linhas entre '*** MARKER ***' e o próximo marcador '***'."""
    start = None
    end = len(lines)
    for i, l in enumerate(lines):
        if l.strip().startswith(marker):
            start = i + 1
        elif start is not None and l.strip().startswith("***"):
            end = i
            break
    return lines[start:end] if start is not None else []


def compute_hand_stats(hand_text):
    lines = hand_text.splitlines()
    players_in_hand = get_players_in_hand(hand_text)
    positions = compute_positions(hand_text)

    preflop_lines = extract_street_lines(lines, "*** HOLE CARDS ***")

    vpip_players = set()
    pfr_players = set()
    three_bet_opportunity_players = set()
    three_bet_players = set()
    faced_3bet_players = set()
    folded_to_3bet_players = set()
    four_bet_opportunity_players = set()
    four_bet_players = set()
    faced_4bet_players = set()
    folded_to_4bet_players = set()
    rfi_opportunity_players = set()
    rfi_players = set()
    faced_steal_players = set()
    folded_to_steal_players = set()
    resteal_players = set()

    acted_players = set()
    limped = False
    steal_raiser = None

    raise_count = 0
    raisers_by_level = {}
    already_had_3bet_opportunity = set()
    already_had_4bet_opportunity = set()
    pending_opener = None
    waiting_for_opener_reaction = False
    pending_3better = None
    waiting_for_3better_reaction = False
    folded_preflop_players = set()

    for line in preflop_lines:
        m = ACTION_RE.match(line.strip())
        if not m:
            continue
        player, action = m.group(1).strip(), m.group(2)

        # oportunidade de 3bet: exatamente 1 raise na mesa, e não fui eu quem fez
        if (raise_count == 1
                and player != raisers_by_level.get(1)
                and player not in already_had_3bet_opportunity):
            three_bet_opportunity_players.add(player)
            already_had_3bet_opportunity.add(player)

        # oportunidade de 4bet: exatamente 2 raises na mesa (um 3bet já feito), e não fui eu
        if (raise_count == 2
                and player != raisers_by_level.get(2)
                and player not in already_had_4bet_opportunity):
            four_bet_opportunity_players.add(player)
            already_had_4bet_opportunity.add(player)

        # primeira ação pré-flop do jogador: RFI, steal e defesa dos blinds
        if player not in acted_players:
            acted_players.add(player)
            if raise_count == 0 and not limped:
                rfi_opportunity_players.add(player)
                if action == "raises":
                    rfi_players.add(player)
                    if positions.get(player) in STEAL_POSITIONS:
                        steal_raiser = player
            elif (steal_raiser is not None and raise_count == 1
                    and player != steal_raiser
                    and positions.get(player) in ("SB", "BB")):
                faced_steal_players.add(player)
                if action == "folds":
                    folded_to_steal_players.add(player)
                elif action == "raises":
                    resteal_players.add(player)
        if action == "calls" and raise_count == 0:
            limped = True

        if action in ("calls", "bets"):
            vpip_players.add(player)
        elif action == "raises":
            vpip_players.add(player)
            pfr_players.add(player)
            raise_count += 1
            raisers_by_level[raise_count] = player
            if raise_count == 2:  # este raise é um 3bet
                three_bet_players.add(player)
                opener = raisers_by_level.get(1)
                if opener and opener not in faced_3bet_players:
                    faced_3bet_players.add(opener)
                    waiting_for_opener_reaction = True
                    pending_opener = opener
            elif raise_count == 3:  # este raise é um 4bet
                four_bet_players.add(player)
                three_better = raisers_by_level.get(2)
                if three_better and three_better not in faced_4bet_players:
                    faced_4bet_players.add(three_better)
                    waiting_for_3better_reaction = True
                    pending_3better = three_better
        elif action == "folds":
            folded_preflop_players.add(player)

        # reação do opener a um 3bet que ele sofreu
        if waiting_for_opener_reaction and player == pending_opener:
            if action == "folds":
                folded_to_3bet_players.add(player)
            waiting_for_opener_reaction = False

        # reação do 3bettor a um 4bet que ele sofreu
        if waiting_for_3better_reaction and player == pending_3better:
            if action == "folds":
                folded_to_4bet_players.add(player)
            waiting_for_3better_reaction = False

    preflop_aggressor = raisers_by_level[max(raisers_by_level)] if raisers_by_level else None

    # -----------------------------------------------------------------
    # Flop, Turn e River: cbet, fold to cbet, raise vs cbet, fold to cbet raise
    # -----------------------------------------------------------------
    flop_lines = extract_street_lines(lines, "*** FLOP ***")
    turn_lines = extract_street_lines(lines, "*** TURN ***")
    river_lines = extract_street_lines(lines, "*** RIVER ***")

    flop_result, flop_aggressor = parse_street_cbet(flop_lines, preflop_aggressor)
    turn_result, turn_aggressor = parse_street_cbet(turn_lines, flop_aggressor)
    river_result, _river_aggressor = parse_street_cbet(river_lines, turn_aggressor)

    # -----------------------------------------------------------------
    # Agressão pós-flop: conta bets/raises/calls em flop+turn+river
    # -----------------------------------------------------------------
    postflop_action_counts = {}
    for street_lines in (flop_lines, turn_lines, river_lines):
        for line in street_lines:
            m = ACTION_RE.match(line.strip())
            if not m:
                continue
            player, action = m.group(1).strip(), m.group(2)
            if action not in ("bets", "raises", "calls"):
                continue
            postflop_action_counts.setdefault(player, {"bets": 0, "raises": 0, "calls": 0})
            postflop_action_counts[player][action] += 1

    # -----------------------------------------------------------------
    # Went to Showdown / Win at Showdown / Won When Saw Flop (WWSF)
    #
    # Usamos a seção "*** SUMMARY ***", que tem uma anotação bem padronizada
    # por jogador (uma linha por assento): ex.
    #   "Seat 6: Frank showed [Ac Kd] and won ($6.30) with a pair of Kings"
    #   "Seat 2: Bob (small blind) mucked [2h 3h]"
    #   "Seat 5: Hero collected ($0.95)"   <- ganhou sem showdown
    #   "Seat 1: Alice folded before Flop"
    # -----------------------------------------------------------------
    saw_flop_players = (players_in_hand - folded_preflop_players) if flop_lines else set()

    summary_lines = extract_street_lines(lines, "*** SUMMARY ***")
    went_to_showdown_players = set()
    won_at_showdown_players = set()
    won_hand_players = set()  # qualquer vitória, com ou sem showdown (usado no WWSF)
    SUMMARY_SEAT_RE = re.compile(r"^Seat \d+: (.+?) (?:\([^)]*\) )?(folded|mucked|showed|collected)\b")
    for line in summary_lines:
        line = line.strip()
        m = SUMMARY_SEAT_RE.match(line)
        if not m:
            continue
        name, verb = m.group(1).strip(), m.group(2)
        if verb in ("showed", "mucked"):
            went_to_showdown_players.add(name)
            if verb == "showed" and " and won" in line:
                won_at_showdown_players.add(name)
                won_hand_players.add(name)
        elif verb == "collected":
            won_hand_players.add(name)
        # "folded" = não chegou ao showdown

    return {
        "players_in_hand": players_in_hand,
        "vpip_players": vpip_players,
        "pfr_players": pfr_players,
        "three_bet_opportunity_players": three_bet_opportunity_players,
        "three_bet_players": three_bet_players,
        "faced_3bet_players": faced_3bet_players,
        "folded_to_3bet_players": folded_to_3bet_players,
        "four_bet_opportunity_players": four_bet_opportunity_players,
        "four_bet_players": four_bet_players,
        "faced_4bet_players": faced_4bet_players,
        "folded_to_4bet_players": folded_to_4bet_players,
        "cbet_opportunity_players": flop_result["opportunity"],
        "cbet_players": flop_result["cbet"],
        "faced_cbet_opp_players": flop_result["faced_cbet_opp"],
        "fold_to_cbet_players": flop_result["fold_to_cbet"],
        "raise_vs_cbet_players": flop_result["raise_vs_cbet"],
        "faced_raise_vs_cbet_players": flop_result["faced_raise_vs_cbet"],
        "fold_to_raise_vs_cbet_players": flop_result["fold_to_cbet_raise"],
        "cbet_turn_opportunity_players": turn_result["opportunity"],
        "cbet_turn_players": turn_result["cbet"],
        "faced_cbet_turn_opp_players": turn_result["faced_cbet_opp"],
        "fold_to_cbet_turn_players": turn_result["fold_to_cbet"],
        "raise_vs_cbet_turn_players": turn_result["raise_vs_cbet"],
        "faced_raise_vs_cbet_turn_players": turn_result["faced_raise_vs_cbet"],
        "fold_to_raise_vs_cbet_turn_players": turn_result["fold_to_cbet_raise"],
        "cbet_river_opportunity_players": river_result["opportunity"],
        "cbet_river_players": river_result["cbet"],
        "faced_cbet_river_opp_players": river_result["faced_cbet_opp"],
        "fold_to_cbet_river_players": river_result["fold_to_cbet"],
        "raise_vs_cbet_river_players": river_result["raise_vs_cbet"],
        "faced_raise_vs_cbet_river_players": river_result["faced_raise_vs_cbet"],
        "fold_to_raise_vs_cbet_river_players": river_result["fold_to_cbet_raise"],
        "postflop_action_counts": postflop_action_counts,
        "saw_flop_players": saw_flop_players,
        "went_to_showdown_players": went_to_showdown_players,
        "won_at_showdown_players": won_at_showdown_players,
        "won_hand_players": won_hand_players,
        "positions": positions,
        "rfi_opportunity_players": rfi_opportunity_players,
        "rfi_players": rfi_players,
        "faced_steal_players": faced_steal_players,
        "folded_to_steal_players": folded_to_steal_players,
        "resteal_players": resteal_players,
    }

# teste com a mão 2 do exemplo: Frank abre, Bob 3beta, Frank folda pro 3bet
compute_hand_stats(sample_hands[1])


In [ ]:
# teste com a mão 3: Frank dá cbet no flop, Alice desiste (fold to cbet)
compute_hand_stats(sample_hands[2])


In [ ]:
# teste com a mão 4: Alice dá cbet, Carol dá raise, Alice desiste (fold to cbet raise)
compute_hand_stats(sample_hands[3])


In [ ]:
# teste com a mão 5: Frank dá cbet no flop, Bob paga; no turn Frank dá cbet
# de novo (segunda barrel) e Bob desiste
compute_hand_stats(sample_hands[4])


In [ ]:
# teste com a mão 6: Bob vê o flop de graça, aposta, Carol desiste - sem showdown
compute_hand_stats(sample_hands[5])


In [ ]:
# teste com a mão 7: vai a showdown, Frank ganha mostrando o par maior
compute_hand_stats(sample_hands[6])


In [ ]:
# teste com a mão 8: Frank abre, Alice dá 3bet, Frank dá 4bet, Alice desiste
compute_hand_stats(sample_hands[7])


In [ ]:
# teste com a mão 9: Hero abre, Frank paga, Hero aposta no flop, Frank desiste (WWSF, sem showdown)
compute_hand_stats(sample_hands[8])


In [ ]:
# teste com a mão 10: Hero abre do CO (steal), Alice no SB dá resteal, Bob (BB) age depois do 3bet
compute_hand_stats(sample_hands[9])


In [ ]:
# teste com a mão 11: Dave rouba do botão, Hero (SB) folda, Frank (BB) paga
compute_hand_stats(sample_hands[10])


Confira o resultado acima: `three_bet_players` deve ter o Bob, e
`folded_to_3bet_players` deve ter o Frank — exatamente o que aconteceu na mão.

## Passo 4 — Somando as estatísticas de várias mãos por jogador

Agora precisamos acumular esses conjuntos (que são por mão) em contadores
totais por jogador, mão após mão.


In [ ]:
# Cada stat é "quantas vezes X aconteceu" sobre "quantas vezes teve a chance
# de acontecer" — isso é só o mapeamento de cada stat pro par de conjuntos
# de jogadores que já calculamos em compute_hand_stats().
FIELD_MAP = [
    ("players_in_hand", "hands"),
    ("vpip_players", "vpip"),
    ("pfr_players", "pfr"),
    ("rfi_opportunity_players", "rfi_opp"),
    ("rfi_players", "rfi"),
    ("three_bet_opportunity_players", "three_bet_opp"),
    ("three_bet_players", "three_bet"),
    ("faced_3bet_players", "faced_3bet_opp"),
    ("folded_to_3bet_players", "faced_3bet_fold"),
    ("four_bet_opportunity_players", "four_bet_opp"),
    ("four_bet_players", "four_bet"),
    ("faced_4bet_players", "faced_4bet_opp"),
    ("folded_to_4bet_players", "faced_4bet_fold"),
    ("cbet_opportunity_players", "cbet_flop_opp"),
    ("cbet_players", "cbet_flop"),
    ("faced_cbet_opp_players", "faced_cbet_flop_opp"),
    ("fold_to_cbet_players", "fold_to_cbet_flop"),
    ("raise_vs_cbet_players", "raise_vs_cbet_flop"),
    ("faced_raise_vs_cbet_players", "faced_raise_vs_cbet_opp"),
    ("fold_to_raise_vs_cbet_players", "fold_to_cbet_raise"),
    ("cbet_turn_opportunity_players", "cbet_turn_opp"),
    ("cbet_turn_players", "cbet_turn"),
    ("faced_cbet_turn_opp_players", "faced_cbet_turn_opp"),
    ("fold_to_cbet_turn_players", "fold_to_cbet_turn"),
    ("raise_vs_cbet_turn_players", "raise_vs_cbet_turn"),
    ("faced_raise_vs_cbet_turn_players", "faced_raise_vs_cbet_turn_opp"),
    ("fold_to_raise_vs_cbet_turn_players", "fold_to_cbet_turn_raise"),
    ("cbet_river_opportunity_players", "cbet_river_opp"),
    ("cbet_river_players", "cbet_river"),
    ("faced_cbet_river_opp_players", "faced_cbet_river_opp"),
    ("fold_to_cbet_river_players", "fold_to_cbet_river"),
    ("raise_vs_cbet_river_players", "raise_vs_cbet_river"),
    ("faced_raise_vs_cbet_river_players", "faced_raise_vs_cbet_river_opp"),
    ("fold_to_raise_vs_cbet_river_players", "fold_to_cbet_river_raise"),
    ("saw_flop_players", "saw_flop"),
    ("went_to_showdown_players", "wtsd"),
    ("won_at_showdown_players", "won_showdown"),
    ("won_hand_players", "won_hand"),
]
POSTFLOP_COUNT_FIELDS = ["postflop_bets", "postflop_raises", "postflop_calls"]
BASE_FIELDS = [field for _, field in FIELD_MAP] + POSTFLOP_COUNT_FIELDS


def empty_player_stats():
    stats = {field: 0 for field in BASE_FIELDS}
    for field in BASE_FIELDS:
        for pos in POSITIONS:
            stats[f"{field}_{pos}"] = 0
    # defesa dos blinds contra steal (só existe pra SB/BB)
    for pos in ("SB", "BB"):
        for campo in ("steal_faced", "steal_fold", "steal_resteal"):
            stats[f"{campo}_{pos}"] = 0
    return stats

def apply_hand_stats(players_db, stats):
    positions = stats["positions"]

    # Cada (conjunto_de_jogadores, campo) gera automaticamente duas versões:
    # o contador geral ("vpip") e, se o jogador tinha posição conhecida
    # naquela mão, o contador por posição ("vpip_UTG", "vpip_BTN", ...).
    # É assim que QUALQUER stat da FIELD_MAP ganha a versão por posição de
    # graça, sem precisar escrever isso stat por stat.
    for set_key, field in FIELD_MAP:
        for player in stats[set_key]:
            players_db.setdefault(player, empty_player_stats())
            players_db[player][field] += 1
            pos = positions.get(player)
            if pos:
                players_db[player][f"{field}_{pos}"] += 1

    # agressão pós-flop: aqui somamos contagens, não "esteve na mão" — mesma
    # ideia, geral + por posição
    for player, counts in stats["postflop_action_counts"].items():
        players_db.setdefault(player, empty_player_stats())
        pos = positions.get(player)
        for action, field in (("bets", "postflop_bets"), ("raises", "postflop_raises"), ("calls", "postflop_calls")):
            players_db[player][field] += counts[action]
            if pos:
                players_db[player][f"{field}_{pos}"] += counts[action]

    # defesa dos blinds contra steal: só faz sentido pra quem estava no SB/BB
    blind_map = [
        ("faced_steal_players", "steal_faced"),
        ("folded_to_steal_players", "steal_fold"),
        ("resteal_players", "steal_resteal"),
    ]
    for set_key, campo in blind_map:
        for player in stats[set_key]:
            pos = positions.get(player)
            if pos in ("SB", "BB"):
                players_db.setdefault(player, empty_player_stats())
                players_db[player][f"{campo}_{pos}"] += 1

# teste: processa as onze mãos de exemplo e mostra o acumulado
players_db_teste = {}
for hand_text in sample_hands:
    apply_hand_stats(players_db_teste, compute_hand_stats(hand_text))

players_db_teste


## Passo 5 — Lendo os arquivos de verdade (e só o que é novo)

Com 30 mil mãos acumuladas por jogador, não faz sentido reprocessar tudo toda
vez. A solução: guardar, para cada arquivo, até que byte já lemos da última
vez (`poker_stats_db.json`). Da próxima vez, só lemos o que foi adicionado
depois disso.

**Importante:** isso cria um problema quando a gente adiciona uma stat nova
que depende de reler mãos antigas (foi o que aconteceu com Agressão
Pós-Flop, WTSD e Win at Showdown — todo mundo apareceu com `None` porque as
mãos antigas já tinham sido "lidas" e nunca mais eram revisitadas, então
nunca passavam pelo código novo). A solução: um número de **versão do
esquema** (`SCHEMA_VERSION`). Toda vez que eu adicionar uma stat que precisa
reler o histórico, esse número sobe — e o `load_db()` detecta que a versão
salva no disco está desatualizada e reprocessa tudo do zero automaticamente
(demora um pouco mais nessa vez, mas só nessa).


In [ ]:
DB_FILE = "poker_stats_db.json"
SCHEMA_VERSION = 6  # aumente este número sempre que adicionar/corrigir algo que precise reler mãos antigas

def load_db():
    if os.path.exists(DB_FILE):
        with open(DB_FILE, "r", encoding="utf-8") as f:
            db = json.load(f)
    else:
        db = {"processed_files": {}, "players": {}, "schema_version": SCHEMA_VERSION}

    if db.get("schema_version") != SCHEMA_VERSION:
        print(f"Esquema de dados mudou (v{db.get('schema_version')} -> v{SCHEMA_VERSION}). "
              f"Reprocessando todo o histórico do zero, isso pode demorar um pouco...")
        db = {"processed_files": {}, "players": {}, "schema_version": SCHEMA_VERSION}

    # migração extra de segurança: preenche com 0 qualquer campo que ainda não exista
    template = empty_player_stats()
    for stats in db["players"].values():
        for field, default in template.items():
            stats.setdefault(field, default)

    return db

def save_db(db):
    db["schema_version"] = SCHEMA_VERSION
    with open(DB_FILE, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)

def process_file(path, db):
    """Lê só as mãos novas de um arquivo (a partir do byte onde paramos da última vez)."""
    key = str(path)
    prev_size = db["processed_files"].get(key, 0)
    size = os.path.getsize(path)
    if size <= prev_size:
        return 0  # nada novo neste arquivo

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        f.seek(prev_size)
        new_text = f.read()

    hands = [h for h in HAND_SPLIT_RE.split(new_text) if h.strip().startswith("PokerStars")]
    for hand_text in hands:
        apply_hand_stats(db["players"], compute_hand_stats(hand_text))

    db["processed_files"][key] = size
    return len(hands)

def find_hand_history_files(root):
    root = Path(root)
    if root.is_file():
        return [root]
    return sorted(root.rglob("*.txt"))


## Passo 6 — Apontando para a sua pasta real

Troque o caminho abaixo pela pasta de hand history do PokerStars (geralmente
`C:\\Users\\SEU_USUARIO\\Documents\\HandHistory\\SEU_NICK`). Rode esta célula
sempre que quiser atualizar os stats — como só lê o que é novo, é rápido
mesmo com uma base grande.

**Atenção:** hand histories de verdade do PokerStars mostram o seu nick real
nas linhas `Seat N: ...` (não aparece como "Hero" — isso só acontece na linha
`Dealt to Hero`, que eu simplifiquei no exemplo). Preencha `HERO_NAME` abaixo
com o seu nick de PokerStars — vamos usar isso mais adiante pra identificar
você mesmo e excluir da lista de oponentes.


In [ ]:
HAND_HISTORY_FOLDER = r"C:\Users\SEU_USUARIO\Documents\HandHistory\SEU_NICK"
HERO_NAME = "SEU_NICK_AQUI"  # seu nick de PokerStars, exatamente como aparece no hand history

db = load_db()

if os.path.exists(HAND_HISTORY_FOLDER):
    files = find_hand_history_files(HAND_HISTORY_FOLDER)
    total_new_hands = sum(process_file(p, db) for p in files)
    save_db(db)
    print(f"Arquivos verificados: {len(files)}")
    print(f"Mãos novas processadas: {total_new_hands}")
    print(f"Jogadores no banco: {len(db['players'])}")
else:
    print("Pasta não encontrada — ajuste HAND_HISTORY_FOLDER acima.")
    print("Por enquanto, vamos usar os dados de exemplo para ver a tabela funcionando:")
    db["players"] = players_db_teste


## Passo 7 — Tabela final

Simples, via pandas: uma linha por jogador, calculando os percentuais a
partir dos contadores brutos. `None`/`NaN` aparece quando o jogador nunca
teve a oportunidade (ex: nunca abriu o pote, então não tem como medir Fold
to 3Bet; ou nunca chegou ao flop como agressor, então não tem Cbet%).


In [ ]:
def pct(numerador, denominador):
    return round(100 * numerador / denominador, 1) if denominador else None

def af(bets, raises, calls):
    return round((bets + raises) / calls, 2) if calls else None

# Cada stat "normal" é só numerador/denominador — como o dicionário de cada
# jogador já tem a versão geral ("vpip") e a versão por posição
# ("vpip_UTG", "vpip_BTN", ...), construir a tabela com TODAS as posições
# de cada stat é só repetir esse cálculo pra cada posição.
STAT_DEFS = [
    ("vpip", "VPIP%", "vpip", "hands"),
    ("pfr", "PFR%", "pfr", "hands"),
    ("rfi", "RFI%", "rfi", "rfi_opp"),
    ("3bet", "3Bet%", "three_bet", "three_bet_opp"),
    ("fold_to_3bet", "Fold to 3Bet%", "faced_3bet_fold", "faced_3bet_opp"),
    ("4bet", "4Bet%", "four_bet", "four_bet_opp"),
    ("fold_to_4bet", "Fold to 4Bet%", "faced_4bet_fold", "faced_4bet_opp"),
    ("cbet_flop", "Cbet Flop%", "cbet_flop", "cbet_flop_opp"),
    ("fold_to_cbet_flop", "Fold to Cbet Flop%", "fold_to_cbet_flop", "faced_cbet_flop_opp"),
    ("cbet_raise_flop", "Raise vs Cbet Flop%", "raise_vs_cbet_flop", "faced_cbet_flop_opp"),
    ("fold_to_cbet_raise_flop", "Fold to Cbet Raise Flop%", "fold_to_cbet_raise", "faced_raise_vs_cbet_opp"),
    ("cbet_turn", "Cbet Turn%", "cbet_turn", "cbet_turn_opp"),
    ("fold_to_cbet_turn", "Fold to Cbet Turn%", "fold_to_cbet_turn", "faced_cbet_turn_opp"),
    ("cbet_raise_turn", "Raise vs Cbet Turn%", "raise_vs_cbet_turn", "faced_cbet_turn_opp"),
    ("fold_to_cbet_raise_turn", "Fold to Cbet Raise Turn%", "fold_to_cbet_turn_raise", "faced_raise_vs_cbet_turn_opp"),
    ("cbet_river", "Cbet River%", "cbet_river", "cbet_river_opp"),
    ("fold_to_cbet_river", "Fold to Cbet River%", "fold_to_cbet_river", "faced_cbet_river_opp"),
    ("cbet_raise_river", "Raise vs Cbet River%", "raise_vs_cbet_river", "faced_cbet_river_opp"),
    ("fold_to_cbet_raise_river", "Fold to Cbet Raise River%", "fold_to_cbet_river_raise", "faced_raise_vs_cbet_river_opp"),
    ("wtsd", "Went to Showdown%", "wtsd", "saw_flop"),
    ("win_at_showdown", "Win at Showdown%", "won_showdown", "wtsd"),
    ("wwsf", "Won When Saw Flop%", "won_hand", "saw_flop"),
]
# rótulo de cada stat (chave curta -> nome de exibição) — usado também pela
# tabela expansível do Passo 7c
STAT_LABELS = {key: label for key, label, _, _ in STAT_DEFS}
STAT_LABELS["af_pos_flop"] = "Agressão Pós-Flop (AF)"

rows = []
for name, s in db["players"].items():
    if s["hands"] == 0:
        continue
    row = {"jogador": name, "maos": s["hands"]}

    for key, label, num_field, den_field in STAT_DEFS:
        row[f"{key}_%"] = pct(s[num_field], s[den_field])
        for pos in POSITIONS:
            row[f"{key}_{pos}_%"] = pct(s.get(f"{num_field}_{pos}", 0), s.get(f"{den_field}_{pos}", 0))

    row["af_pos_flop"] = af(s["postflop_bets"], s["postflop_raises"], s["postflop_calls"])
    for pos in POSITIONS:
        row[f"af_pos_flop_{pos}"] = af(
            s.get(f"postflop_bets_{pos}", 0),
            s.get(f"postflop_raises_{pos}", 0),
            s.get(f"postflop_calls_{pos}", 0),
        )

    for pos in POSITIONS:
        row[f"maos_{pos}"] = s[f"hands_{pos}"]

    # ATS = RFI somado das posições de steal (CO, BTN, SB)
    ats_ok = sum(s[f"rfi_{p}"] for p in STEAL_POSITIONS)
    ats_opp = sum(s[f"rfi_opp_{p}"] for p in STEAL_POSITIONS)
    row["ats_%"] = pct(ats_ok, ats_opp)

    # defesa dos blinds (SB, BB e combinado)
    faced = {p: s[f"steal_faced_{p}"] for p in ("SB", "BB")}
    folded = {p: s[f"steal_fold_{p}"] for p in ("SB", "BB")}
    resteal = {p: s[f"steal_resteal_{p}"] for p in ("SB", "BB")}
    row["fold_to_steal_%"] = pct(sum(folded.values()), sum(faced.values()))
    row["fold_to_steal_SB_%"] = pct(folded["SB"], faced["SB"])
    row["fold_to_steal_BB_%"] = pct(folded["BB"], faced["BB"])
    row["resteal_%"] = pct(sum(resteal.values()), sum(faced.values()))
    row["resteal_SB_%"] = pct(resteal["SB"], faced["SB"])
    row["resteal_BB_%"] = pct(resteal["BB"], faced["BB"])

    rows.append(row)

df = pd.DataFrame(rows).sort_values("maos", ascending=False).reset_index(drop=True)
df


## Passo 7b — Escolha quais stats aparecem na tabela

A tabela `df` acima tem **todas** as stats calculadas. Aqui você escolhe quais
quer ver — a lista `COLUNAS_ESCOLHIDAS` define as colunas exibidas (além de
`jogador` e `maos`, que sempre aparecem). A função `visao()` aplica essa
escolha, e é usada nos passos seguintes (filtro por jogador, mesa atual,
menu de mesas), então basta escolher uma vez.

**Jeito mais simples (sem widget):** edite a lista abaixo — apague/comente as
linhas das stats que não quer — e rode a célula.


In [ ]:
# Stats disponíveis, agrupadas por categoria: {coluna: nome de exibição}
STAT_GROUPS = {
    "Pré-flop": {
        "vpip_%": "VPIP%",
        "pfr_%": "PFR%",
        "rfi_%": "RFI%",
        "3bet_%": "3Bet%",
        "fold_to_3bet_%": "Fold to 3Bet%",
        "4bet_%": "4Bet%",
        "fold_to_4bet_%": "Fold to 4Bet%",
    },
    "Cbet — Flop": {
        "cbet_flop_%": "Cbet Flop%",
        "fold_to_cbet_flop_%": "Fold to Cbet Flop%",
        "cbet_raise_flop_%": "Raise vs Cbet Flop%",
        "fold_to_cbet_raise_flop_%": "Fold to Cbet Raise Flop%",
    },
    "Cbet — Turn": {
        "cbet_turn_%": "Cbet Turn%",
        "fold_to_cbet_turn_%": "Fold to Cbet Turn%",
        "cbet_raise_turn_%": "Raise vs Cbet Turn%",
        "fold_to_cbet_raise_turn_%": "Fold to Cbet Raise Turn%",
    },
    "Cbet — River": {
        "cbet_river_%": "Cbet River%",
        "fold_to_cbet_river_%": "Fold to Cbet River%",
        "cbet_raise_river_%": "Raise vs Cbet River%",
        "fold_to_cbet_raise_river_%": "Fold to Cbet Raise River%",
    },
    "Pós-flop e showdown": {
        "af_pos_flop": "Agressão Pós-Flop (AF)",
        "wtsd_%": "Went to Showdown%",
        "win_at_showdown_%": "Win at Showdown%",
        "wwsf_%": "Won When Saw Flop%",
    },
    "Steal e defesa dos blinds": {
        "ats_%": "ATS% (steal CO/BTN/SB)",
        "fold_to_steal_%": "Fold to Steal% (blinds)",
        "fold_to_steal_SB_%": "Fold to Steal% SB",
        "fold_to_steal_BB_%": "Fold to Steal% BB",
        "resteal_%": "Resteal% (blinds)",
        "resteal_SB_%": "Resteal% SB",
        "resteal_BB_%": "Resteal% BB",
    },
    "Por posição — VPIP": {
        "vpip_UTG_%": "VPIP% UTG",
        "vpip_MP_%": "VPIP% MP",
        "vpip_CO_%": "VPIP% CO",
        "vpip_BTN_%": "VPIP% BTN",
        "vpip_SB_%": "VPIP% SB",
        "vpip_BB_%": "VPIP% BB",
    },
    "Por posição — PFR": {
        "pfr_UTG_%": "PFR% UTG",
        "pfr_MP_%": "PFR% MP",
        "pfr_CO_%": "PFR% CO",
        "pfr_BTN_%": "PFR% BTN",
        "pfr_SB_%": "PFR% SB",
        "pfr_BB_%": "PFR% BB",
    },
    "Por posição — RFI (raise first in)": {
        "rfi_UTG_%": "RFI% UTG",
        "rfi_MP_%": "RFI% MP",
        "rfi_CO_%": "RFI% CO",
        "rfi_BTN_%": "RFI% BTN",
        "rfi_SB_%": "RFI% SB",
    },
    "Por posição — nº de mãos": {
        "maos_UTG": "Mãos em UTG",
        "maos_MP": "Mãos em MP",
        "maos_CO": "Mãos em CO",
        "maos_BTN": "Mãos em BTN",
        "maos_SB": "Mãos em SB",
        "maos_BB": "Mãos em BB",
    },
}

# lista única com todas as stats (usada pelo seletor e pelo widget)
STAT_COLUMN_LABELS = {c: label for grupo in STAT_GROUPS.values() for c, label in grupo.items()}


# >>> EDITE AQUI: deixe só as stats que quer ver (apague ou comente as outras) <<<
COLUNAS_ESCOLHIDAS = [
    # --- Pré-flop ---
    "vpip_%",
    "pfr_%",
    "3bet_%",
    "fold_to_3bet_%",
    "4bet_%",
    "fold_to_4bet_%",
    # --- Cbet — Flop ---
    "cbet_flop_%",
    "fold_to_cbet_flop_%",
    "cbet_raise_flop_%",
    "fold_to_cbet_raise_flop_%",
    # --- Cbet — Turn ---
    "cbet_turn_%",
    "fold_to_cbet_turn_%",
    # "cbet_raise_turn_%",
    # "fold_to_cbet_raise_turn_%",
    # --- Cbet — River ---
    # "cbet_river_%",
    # "fold_to_cbet_river_%",
    # "cbet_raise_river_%",
    # "fold_to_cbet_raise_river_%",
    # --- Pós-flop e showdown ---
    "af_pos_flop",
    "wtsd_%",
    "win_at_showdown_%",
    "wwsf_%",
    # --- Steal e defesa dos blinds ---
    # "ats_%",
    # "fold_to_steal_%",
    # "fold_to_steal_SB_%",
    # "fold_to_steal_BB_%",
    # "resteal_%",
    # "resteal_SB_%",
    # "resteal_BB_%",
    # --- Por posição — VPIP ---
    # "vpip_UTG_%",
    # "vpip_MP_%",
    # "vpip_CO_%",
    # "vpip_BTN_%",
    # "vpip_SB_%",
    # "vpip_BB_%",
    # --- Por posição — PFR ---
    # "pfr_UTG_%",
    # "pfr_MP_%",
    # "pfr_CO_%",
    # "pfr_BTN_%",
    # "pfr_SB_%",
    # "pfr_BB_%",
    # --- Por posição — RFI (raise first in) ---
    # "rfi_UTG_%",
    # "rfi_MP_%",
    # "rfi_CO_%",
    # "rfi_BTN_%",
    # "rfi_SB_%",
    # --- Por posição — nº de mãos ---
    # "maos_UTG",
    # "maos_MP",
    # "maos_CO",
    # "maos_BTN",
    # "maos_SB",
    # "maos_BB",
]


def visao(tabela):
    """Devolve a tabela só com jogador, maos e as stats em COLUNAS_ESCOLHIDAS."""
    colunas = ["jogador", "maos"] + [c for c in COLUNAS_ESCOLHIDAS if c in tabela.columns]
    return tabela[colunas]


visao(df)


**Alternativa com caixas de seleção (widget):** marque/desmarque as stats
abaixo e a tabela se atualiza na hora. Se aparecer "Loading widget..." no seu
Jupyter, ignore esta célula e use a lista acima — o resultado é o mesmo.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

checkboxes = {
    coluna: widgets.Checkbox(
        value=(coluna in COLUNAS_ESCOLHIDAS),
        description=rotulo,
        indent=False,
        layout=widgets.Layout(width="260px"),
    )
    for coluna, rotulo in STAT_COLUMN_LABELS.items()
}
saida_tabela = widgets.Output()
btn_todas = widgets.Button(description="Marcar todas")
btn_nenhuma = widgets.Button(description="Desmarcar todas")


def aplicar_selecao(_=None):
    COLUNAS_ESCOLHIDAS[:] = [c for c, cb in checkboxes.items() if cb.value]
    with saida_tabela:
        clear_output()
        display(visao(df))


def marcar(valor):
    def _handler(_):
        for cb in checkboxes.values():
            cb.value = valor
    return _handler


for cb in checkboxes.values():
    cb.observe(aplicar_selecao, names="value")
btn_todas.on_click(marcar(True))
btn_nenhuma.on_click(marcar(False))

# uma seção por categoria, com as caixas de seleção em grade
secoes = []
for grupo, colunas in STAT_GROUPS.items():
    secoes.append(widgets.HTML(f"<b>{grupo}</b>"))
    secoes.append(widgets.GridBox(
        [checkboxes[c] for c in colunas],
        layout=widgets.Layout(grid_template_columns="repeat(3, 270px)"),
    ))

aplicar_selecao()
display(widgets.VBox([widgets.HBox([btn_todas, btn_nenhuma])] + secoes + [saida_tabela]))


## Passo 8 — Filtrar por um jogador específico

Só trocar o nome abaixo (não precisa ser exato — busca por trecho do nome,
sem diferenciar maiúsculas/minúsculas) e rodar a célula.


In [ ]:
NOME_DO_JOGADOR = "Frank"  # <-- troque pelo nome que quiser consultar

visao(df[df["jogador"].str.contains(NOME_DO_JOGADOR, case=False, na=False)])


## Passo 9 — Quem eu estou jogando agora?

A ideia: olhar o arquivo de hand history modificado mais recentemente (esse é
a mesa em que você está jogando agora, já que o PokerStars escreve nele a
cada mão), pegar a última mão desse arquivo e listar quem estava sentado —
esses são seus oponentes atuais. Depois é só filtrar a tabela `df` por esses
nomes.

Isso só funciona com `HAND_HISTORY_FOLDER` (Passo 6) apontando pra pasta de
verdade e com `HERO_NAME` preenchido com o seu nick, pra te excluirmos da
lista de oponentes.


In [ ]:
def get_last_hand_players(folder):
    """Retorna (arquivo_mais_recente, jogadores_da_ultima_mao)."""
    files = find_hand_history_files(folder)
    if not files:
        return None, set()

    most_recent_file = max(files, key=lambda p: p.stat().st_mtime)
    text = most_recent_file.read_text(encoding="utf-8", errors="ignore")
    hands = [h for h in HAND_SPLIT_RE.split(text) if h.strip().startswith("PokerStars")]
    if not hands:
        return most_recent_file, set()

    last_hand = hands[-1]
    return most_recent_file, get_players_in_hand(last_hand)


def show_current_opponents(folder, hero_name, stats_df):
    arquivo, jogadores = get_last_hand_players(folder)
    if arquivo is None:
        print("Nenhum hand history encontrado na pasta configurada.")
        return None

    oponentes = jogadores - {hero_name}
    print(f"Mesa mais recente: {arquivo.name}")
    print(f"Jogadores na última mão: {sorted(jogadores)}")
    print(f"Oponentes (excluindo HERO_NAME='{hero_name}'): {sorted(oponentes)}")

    return visao(stats_df[stats_df["jogador"].isin(oponentes)])


show_current_opponents(HAND_HISTORY_FOLDER, HERO_NAME, df)


## Passo 10 — Mesa atual (menu suspenso + tabela expansível por posição)

Essa é a visão que você usa jogando: escolhe a mesa num menu suspenso (cobre
o caso de jogar 2 ou 3 mesas ao mesmo tempo — cada uma grava no próprio
arquivo, e todas aparecem como opção) e vê, só dos jogadores daquela mesa,
uma tabela onde cada stat tem um resumo sempre visível e, clicando no
cabeçalho, abre as 6 colunas por posição (UTG, MP, CO, BTN, SB, BB) ao lado.

Clique em "🔄 Atualizar mesas ativas" sempre que sentar numa mesa nova (ou
sair de uma). Trocar a seleção no menu já atualiza a tabela na hora. Edite
`PIVOT_STATS` pra escolher quais stats aparecem nela.


In [ ]:
import uuid
from IPython.display import HTML, display

# Por padrão, mostra todas as stats "normais" (as que têm versão por
# posição). As compostas (ATS, fold to steal, resteal) já são inerentemente
# por posição (só existem pra CO/BTN/SB ou SB/BB), então não entram aqui.
PIVOT_STATS = [key for key, _, _, _ in STAT_DEFS] + ["af_pos_flop"]


def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "-"
    if isinstance(v, (int,)) and not isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        return f"{v:.1f}" if abs(v) < 1000 else f"{v:.0f}"
    return str(v)


def render_pivot(tabela, stat_keys=None, positions=POSITIONS):
    """Tabela em HTML: jogador x stats, com uma coluna-resumo sempre visível
    por stat e um grupo de colunas por posição que abre/fecha ao clicar no
    cabeçalho. Não depende de ipywidgets — é só HTML + um pouco de JS."""
    stat_keys = stat_keys if stat_keys is not None else PIVOT_STATS
    tid = uuid.uuid4().hex[:8]  # evita conflito se renderizar mais de uma vez na página

    head_cells = ['<th style="position:sticky; left:0; background:#f0f0f0;">Jogador</th>',
                  '<th>Mãos</th>']
    for key in stat_keys:
        label = STAT_LABELS.get(key, key)
        head_cells.append(
            f'<th class="stat-head" onclick="togglePivot_{tid}(\'{key}\')" '
            f'style="cursor:pointer; white-space:nowrap; background:#e8e8e8;" '
            f'title="Clique pra expandir/recolher por posição">'
            f'{label} <span id="arrow-{tid}-{key}">▸</span></th>'
        )
        for pos in positions:
            head_cells.append(
                f'<th data-pivot="{tid}" data-group="{key}" '
                f'style="display:none; white-space:nowrap; background:#f7f7f7;">{pos}</th>'
            )

    body_rows = []
    for _, r in tabela.iterrows():
        cells = [f'<td style="position:sticky; left:0; background:#fff; font-weight:bold;">{r["jogador"]}</td>',
                 f'<td>{_fmt(r.get("maos"))}</td>']
        for key in stat_keys:
            col = f"{key}_%" if f"{key}_%" in tabela.columns else key
            cells.append(f'<td>{_fmt(r.get(col))}</td>')
            for pos in positions:
                pos_col = f"{key}_{pos}_%" if f"{key}_{pos}_%" in tabela.columns else f"{key}_{pos}"
                cells.append(
                    f'<td data-pivot="{tid}" data-group="{key}" style="display:none;">'
                    f'{_fmt(r.get(pos_col))}</td>'
                )
        body_rows.append("<tr>" + "".join(cells) + "</tr>")

    html = f"""
<div>
  <button onclick="setAllPivot_{tid}(true)">Expandir todas</button>
  <button onclick="setAllPivot_{tid}(false)">Recolher todas</button>
  <div style="overflow-x:auto; max-height:520px; overflow-y:auto; margin-top:6px;">
    <table border="1" style="border-collapse:collapse; font-size:13px;">
      <thead><tr>{''.join(head_cells)}</tr></thead>
      <tbody>{''.join(body_rows)}</tbody>
    </table>
  </div>
</div>
<script>
function togglePivot_{tid}(key) {{
    var els = document.querySelectorAll('[data-pivot="{tid}"][data-group="' + key + '"]');
    if (els.length === 0) return;
    var hidden = els[0].style.display === 'none';
    els.forEach(function(e) {{ e.style.display = hidden ? 'table-cell' : 'none'; }});
    var arrow = document.getElementById('arrow-{tid}-' + key);
    if (arrow) arrow.textContent = hidden ? '▾' : '▸';
}}
function setAllPivot_{tid}(show) {{
    document.querySelectorAll('[data-pivot="{tid}"]').forEach(function(e) {{
        e.style.display = show ? 'table-cell' : 'none';
    }});
    document.querySelectorAll('[id^="arrow-{tid}-"]').forEach(function(a) {{
        a.textContent = show ? '▾' : '▸';
    }});
}}
</script>
"""
    display(HTML(html))


import ipywidgets as widgets
from IPython.display import display, clear_output

TABLE_NAME_RE = re.compile(r"Table '(.+?)'")

def get_table_name(hand_text):
    m = TABLE_NAME_RE.search(hand_text)
    return m.group(1) if m else "Mesa desconhecida"


def get_active_tables(folder, minutes=15, hero_name=""):
    """Uma entrada por arquivo com atividade nos últimos `minutes` minutos —
    cobre o caso de várias mesas abertas ao mesmo tempo."""
    files = find_hand_history_files(folder)
    now = time.time()
    active = []

    for f in files:
        try:
            age_minutes = (now - f.stat().st_mtime) / 60
        except OSError:
            continue
        if age_minutes > minutes:
            continue

        text = f.read_text(encoding="utf-8", errors="ignore")
        hands = [h for h in HAND_SPLIT_RE.split(text) if h.strip().startswith("PokerStars")]
        if not hands:
            continue

        last_hand = hands[-1]
        players = get_players_in_hand(last_hand)
        active.append({
            "file": f,
            "table_name": get_table_name(last_hand),
            "players": players,
            "opponents": players - {hero_name} if hero_name else players,
            "age_minutes": age_minutes,
        })

    active.sort(key=lambda t: t["age_minutes"])
    return active


ACTIVE_MINUTES = 15  # janela de tempo pra considerar uma mesa "ativa"

mesa_dropdown = widgets.Dropdown(description="Mesa:", layout=widgets.Layout(width="500px"))
atualizar_mesas_btn = widgets.Button(description="🔄 Atualizar mesas ativas", button_style="primary")
output_mesa = widgets.Output()


def mostrar_mesa_selecionada(change=None):
    with output_mesa:
        clear_output()
        mesa = mesa_dropdown.value
        if not mesa:
            print(f"Nenhuma mesa ativa encontrada nos últimos {ACTIVE_MINUTES} min.")
            return
        print(f"Mesa: {mesa['table_name']}  |  atualizada há {mesa['age_minutes']:.1f} min")
        render_pivot(df[df["jogador"].isin(mesa["opponents"])])


def atualizar_mesas_ativas(_=None):
    ativas = get_active_tables(HAND_HISTORY_FOLDER, minutes=ACTIVE_MINUTES, hero_name=HERO_NAME)
    if not ativas:
        mesa_dropdown.options = [("Nenhuma mesa ativa", None)]
    else:
        mesa_dropdown.options = [
            (f"{t['table_name']}  ·  há {t['age_minutes']:.1f} min  ·  {len(t['players'])} jogadores", t)
            for t in ativas
        ]
    mostrar_mesa_selecionada()


atualizar_mesas_btn.on_click(atualizar_mesas_ativas)
mesa_dropdown.observe(mostrar_mesa_selecionada, names="value")

atualizar_mesas_ativas()
display(widgets.VBox([atualizar_mesas_btn, mesa_dropdown, output_mesa]))


## Próximos passos (se quiser evoluir)

- Filtrar as stats por **tamanho de amostra** (ex: esconder valores com poucas
  mãos) — as colunas de "nº de mãos por posição" ajudam a avaliar isso.
- Squeeze% e 3bet/fold to 3bet por posição.
- Ordenar por outra coluna: `visao(df).sort_values("vpip_%", ascending=False)`
- Salvar a tabela em Excel: `visao(df).to_excel("stats.xlsx", index=False)`
